# Week 6 practice: your own OpenAlex request

In the browser workspace most of the code was written for you. Here you write it yourself, for a question of your own.

Work in your own copy: duplicate this notebook and add `_mine` to the end of its name.

**Your question.** Choose an institution and a year. What are the 25 most-cited works from that year with an author at that institution, and which fields are they in?

| Institution | OpenAlex ID |
| --- | --- |
| UC Berkeley | `I95457486` |
| UCLA | `I161318765` |
| Stanford University | `I97018004` |
| MIT | `I63966007` |
| Harvard University | `I136199984` |
| University of Oxford | `I40120149` |
| University of Amsterdam | `I887064364` |

Part 1 needs no key.

## 1. Look up your institution

1. Import `requests`.
2. Send a GET request with `requests.get()` to `https://api.openalex.org/institutions/` + your ID. Name the result `response`.
3. Print `response.status_code`.
4. Turn the body into a dictionary: `record = response.json()`.
5. Print `record["display_name"]` and `record["works_count"]`. Is it the institution you chose?

## 2. Ask for the works

1. Make a dictionary named `params` with four keys:
   - `"filter"`: your institution and year, like `"institutions.id:I95457486,publication_year:1999"`
   - `"sort"`: `"cited_by_count:desc"`
   - `"per_page"`: `25`
   - `"cursor"`: `"*"`
2. Send it with `requests.get()`: the endpoint `"https://api.openalex.org/works"`, then `params=params` and `timeout=30`. Name the result `response`.
3. Print `response.status_code` and `response.url`.

Run this cell once. OpenAlex limits the list requests a network can send in a day, and the class shares one network.

## 3. Read an error

1. Write your dictionary again as `bad_params`, with `"per_page"` set to `500`.
2. Send it as in step 2. Name the result `bad_response`, so that `response` keeps the good one.
3. Print `bad_response.status_code` and `bad_response.text`. What does the server say is wrong?

## 4. Save the response

1. Import `json`.
2. Turn the good response into Python data: `data = response.json()`.
3. Save it: `with open("openalex_mine.json", "w") as f:` and, inside, `json.dump(data, f)`.
4. Read it back: `with open("openalex_mine.json") as f:` and, inside, `data = json.load(f)`.

From here on, work from the file. Do not run the request cell again.

## 5. Look at what came back

1. Print `data["meta"]["count"]`: the works that matched.
2. Print `len(data["results"])`: the works in the response.
3. Take the first work: `work = data["results"][0]`. Print `list(work.keys())`. The workspace's copy kept seven of these.
4. Print the name of its field. Go one level at a time: `"primary_topic"`, then `"field"`, then `"display_name"`.

## 6. Make a table

1. Import `pandas` as `pd`.
2. Start an empty list: `rows = []`.
3. Loop over `data["results"]`. For each work, append a dictionary to `rows` with four keys:
   - `"title"`: its `display_name`
   - `"cited_by"`: its `cited_by_count`
   - `"first_author"`: `authorships`, then `[0]`, then `author`, then `display_name`
   - `"field"`: the name of its field, as in step 5
4. Make the table: `table = pd.DataFrame(rows)`.

If the loop stops with a `TypeError`, a work has `None` as its `primary_topic`. Test for that with `is None` first.

## 7. Count the fields

Count the works in each field: `value_counts()` on the `"field"` column of `table`.

## 8. What does the table show?

Someone calls your table "this institution's 25 most important papers of that year", and reads your count as the fields it is best at. In two or three sentences: what do the table and the count show, and what do they not?

*Your sentences:*

## 9. If you have time: the next page

1. Set `params["cursor"]` to `data["meta"]["next_cursor"]`.
2. Send the request again as `response2`, and make `data2` from it with `.json()`.
3. Loop over `data2["results"]` and append to `rows`, as in step 6.
4. Make `table` again. It should have 50 rows. Do the counts change?

## If the request does not work

With no network, or with status 429 (too many requests today), open the workspace's saved response and continue from step 5:

`with open("../../docs/interactives/data/week06/openalex_page1.json") as f:` and, inside, `data = json.load(f)`

It holds five works from 2024 with a UC Berkeley author.

# Part 2: the same request, with a key

Many APIs ask for an API key: a password that says who is sending the request. OpenAlex works without one, and allows more requests a day with one.

Never paste a key into a notebook or an AI chat. Keep it in a file named `.env`, which Git ignores.

This part can be done on another day. In a new session, run your cells from steps 1 and 2 first.

## 10. See where a key shows up

1. Write your dictionary again as `fake_params`, with one more key: `"api_key"`, set to `"not-a-real-key"`.
2. Send it as in step 2. Name the result `fake_response`.
3. Print `fake_response.status_code`, `fake_response.text` and `fake_response.url`.
4. In the next cell, run `fake_response.raise_for_status()`. It stops with an error.

Find the made-up key in the address and in the error message. A message like this is what you might paste into an AI chat to ask for help.

## 11. Get a key and put it in `.env`

`.env` does not exist until you make it. Finder and File Explorer may not show it, so use a terminal.

1. Make a free account at [openalex.org](https://openalex.org).
2. Copy your key from [openalex.org/settings/api-key](https://openalex.org/settings/api-key). Do not paste it into this notebook.
3. Open a terminal in the top folder of the repository: in VS Code, **Terminal > New Terminal**. `pwd` should end in `compss-211a`.
4. Make your `.env` from the template, then open it. Skip the first command if you made a `.env` before.
   - Mac: `cp .env.example .env`, then `open -e .env`
   - Windows: `cp .env.example .env`, then `notepad .env`
5. Put the key after `OPENALEX_API_KEY=`, with no quotes and no spaces. Save and close the file.
6. In the terminal, run `git status`. `.env` must not be in the list.

## 12. Read the key without showing it

1. Import `os`, and `load_dotenv` from `dotenv`.
2. Run `load_dotenv(override=True)`. It reads `.env`.
3. Get the key: `key = os.environ["OPENALEX_API_KEY"]`.
4. Print `len(key)`, not the key.

A `KeyError` means `.env` was not found, or has no `OPENALEX_API_KEY` line. A length of 0 means nothing comes after the `=`.

## 13. Send the key in a header

A header is sent with a request, but is not part of the address.

1. Send your step 2 request again with one more argument: `headers={"Authorization": "Bearer " + key}`. Name the result `key_response`.
2. Print `key_response.status_code` and `key_response.url`.

The made-up key got 401. Yours should get 200, with no key in the address.

## 14. Check before you share

1. Search this notebook for the first few characters of your key, with Ctrl+F or ⌘F. They should not be in any cell or output.
2. If a key does end up in a notebook, a chat or a commit, press **Rotate** on the settings page to replace it.